In [1]:
import os
import glob
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch

# -------------------------------
# GPU 检查（如果有 GPU 则使用 GPU）
# -------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using GPU: GeForce GTX 1650


In [ ]:
import ast
import pandas as pd

# 读取CSV标注数据
annotations_file_path = 'E:/pythonfile/main-task/annotations改.csv'
annotations_df = pd.read_csv(annotations_file_path)
annotations_df['ID'] = annotations_df['ID'].str.replace('.fts', '', regex=False)
#annotations_df.info()
#annotations_df.head()

# 定义安全的转换函数
def safe_literal_eval(val):
    try:
        # 如果是字符串形式的元组，尝试转换为元组
        return ast.literal_eval(val) if isinstance(val, str) else val
    except (ValueError, SyntaxError):
        # 如果发生格式错误或其他异常，返回None
        return None

# 需要处理的列
columns_to_process = ['filter', 'rell_number', 'date', 'hour', 'minute', 'seconds']

# 对每一列应用safe_literal_eval函数，跳过空值
for col in columns_to_process:
    annotations_df[col] = annotations_df[col].apply(safe_literal_eval)

In [ ]:
import os
import pandas as pd
import re
import xml.etree.ElementTree as ET
from xml.dom import minidom

# 读取数据（同上）
df =  annotations_df
bbox_cols = ["filter", "rell_number", "date", "hour", "minute", "seconds"]

# 辅助函数
def parse_bbox_str(bbox_str):
    numbers = re.findall(r'\d+', bbox_str)
    if len(numbers) == 4:
        return list(map(int, numbers))
    else:
        return None

# 分组：按 image_id 分组，每张图片对应一个 XML 文件
grouped = df.groupby("ID")

output_dir = "E:/pythonfile/main-task/image_split/temp_output"
os.makedirs(output_dir, exist_ok=True)

def prettify(elem):
    rough_string = ET.tostring(elem, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent="  ")

for image_id, group in grouped:
    annotation = ET.Element("annotation")
    # 添加 image_id
    ET.SubElement(annotation, "filename").text = image_id
    
    # 遍历所有 bbox 列，按顺序添加目标框
    for col in bbox_cols:
        for val in group[col]:
            if pd.isnull(val):
                continue
            bbox = parse_bbox_str(str(val))
            if bbox is None:
                continue
            obj = ET.SubElement(annotation, "object")
            ET.SubElement(obj, "name").text = col   #设置类别名称
            bndbox = ET.SubElement(obj, "bndbox")
            ET.SubElement(bndbox, "xmin").text = str(bbox[0])
            ET.SubElement(bndbox, "ymin").text = str(bbox[1])
            ET.SubElement(bndbox, "xmax").text = str(bbox[2])
            ET.SubElement(bndbox, "ymax").text = str(bbox[3])
    
    # 保存 XML 文件
    xml_str = prettify(annotation)
    xml_path = os.path.join(output_dir, os.path.splitext(image_id)[0] + ".xml")
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write(xml_str)
    print(f"生成 XML 文件：{xml_path}")


In [ ]:
#导入xml文件代码需要fts_gz_files
import os
import glob
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class FastRCNNDataset(Dataset):
    def __init__(self, fts_gz_files, xml_dir, transform=None, label_mapping=None):
        """
        :param fts_gz_files: 按顺序存放的图像路径列表（无需文件名匹配）
        :param xml_dir: 存放 XML 标注文件的目录
        :param transform: torchvision.transforms 预处理流水线
        :param label_mapping: 类别映射字典，如 {'text': 1}
        """
        self.fts_gz_files = fts_gz_files
        self.transform = transform
        self.label_mapping = label_mapping or {'text': 1}

        # 读取并排序所有 XML 文件
        xml_files = glob.glob(os.path.join(xml_dir, "*.xml"))
        if len(xml_files) != len(self.fts_gz_files):
            raise ValueError(
                f"图像数 ({len(self.fts_gz_files)}) 与 XML 文件数 ({len(xml_files)}) 不一致！"
            )

        # 解析每个 XML 文件，按顺序存入列表
        self.annos = []
        for xml_file in xml_files:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            boxes = []
            labels = []
            for obj in root.findall('object'):
                label = obj.find('name').text
                bnd = obj.find('bndbox')
                xmin = int(bnd.find('xmin').text)
                ymin = int(bnd.find('ymin').text)
                xmax = int(bnd.find('xmax').text)
                ymax = int(bnd.find('ymax').text)
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(label)
            if not boxes:
                # 如果某些图像没有目标，也要占位，以保证索引对齐
                self.annos.append({'boxes': np.zeros((0,4),dtype=np.float32),
                                   'labels': []})
            else:
                self.annos.append({
                    'boxes': np.array(boxes, dtype=np.float32),
                    'labels': labels
                })

    def __len__(self):
        return len(self.fts_gz_files)

    def __getitem__(self, idx):
        # 直接从列表获取图像数据（此处 image 是一个 numpy.ndarray）
        image = self.fts_gz_files[idx]
        print(image)

        # 如果 image 是 NumPy 数组，则转换为 PIL Image
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)

        # 2. 应用预处理
        if self.transform:
            image = self.transform(image)

        # 3. 获取对应的标注
        anno = self.annos[idx]
        boxes = anno['boxes']          # shape: (N,4)
        labels_str = anno['labels']
        labels = np.array(
            [self.label_mapping.get(lbl, 0) for lbl in labels_str],
            dtype=np.int64
        )

        target = {"boxes": boxes, "labels": labels}
        return image, target


xml_dir = "E:/pythonfile/main-task/image_split/temp_output"

# 定义图像预处理
transform = transforms.Compose([
    transforms.Resize((600, 800)),
    transforms.ToTensor(),
    # transforms.Normalize(mean=[...], std=[...])
])

# 类别映射：根据你的 XML 中 <name> 标签的内容设置
label_mapping = {'background': 0, 'filter': 1, 'rell_number': 2, 'date': 3, 'hour': 4, 'minute': 5, 'seconds': 6}

# 创建数据集与 DataLoader
dataset = FastRCNNDataset(
    fts_gz_files=fts_gz_files,
    xml_dir=xml_dir,
    transform=transform,
    label_mapping=label_mapping
)

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

# 检查第一个 batch
for imgs, tgts in dataloader:
    for i, img in enumerate(imgs):
        print(f"Image {i} tensor shape:", img.shape)
        print("  boxes:", tgts[i]['boxes'])
        print("  labels:", tgts[i]['labels'])
    break


In [ ]:
#可视化图像及其对应的标注框。
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def visualize_annotations(image, boxes, labels=None, label_names=None):
    """
    可视化图像及其对应的标注框。
    
    参数:
      image: PIL Image 或 ndarray 格式图像
      boxes: 标注框列表/数组，形状为 (N, 4)，每个框为 [x_center, y_center, width, height]，数值已归一化
      labels: 可选，每个框的类别标签，数组或列表，长度与 boxes 相同
      label_names: 可选，一个字典或列表，对应类别id到类别名称的映射
      
    注意：此处假设图像尺寸与标注时使用的尺寸一致（例如 800×600 或 600×800）。
    """
    # 如果输入图像为 numpy 数组，转换为 PIL
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    
    # 获取图像尺寸
    width, height = image.size

    # 创建绘图窗口
    fig, ax = plt.subplots(1)
    ax.imshow(image, cmap='gray')

    # 遍历所有标注框，转换为像素坐标并绘制矩形框
    for i, box in enumerate(boxes):
        # YOLO格式为归一化值：x_center, y_center, width, height
        x_center, y_center, w, h = box
        # 转换为像素值
        xmin = (x_center - w/2) * width#+0.045
        ymin = (y_center - h/2) * height
        rect_width = w * width
        rect_height = h * height
        
        # 创建矩形patch
        rect = patches.Rectangle((xmin, ymin), rect_width, rect_height, 
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        # 如果有类别标签
        if labels is not None and label_names is not None:
            label = labels[i]
            # 尝试获取类别名称
            if isinstance(label_names, dict):
                label_text = label_names.get(label, str(label))
            elif isinstance(label_names, (list, tuple)):
                label_text = label_names[label]
            else:
                label_text = str(label)
            ax.text(xmin, ymin-5, label_text, color='yellow', fontsize=10, weight='bold')
    
    ax.set_title("标注框可视化")
    plt.axis('off')
    plt.show()

# 示例使用方法：假设你已经获得了一张经过转换的图像，以及对应的标注框 boxes 和标签 labels
# 下面给出一个模拟示例，你可以用实际数据替换

# 模拟数据（归一化后的YOLO格式）
'''
boxes = [
    [0.5, 0.5, 0.3, 0.2],  # 居中一个框
    [0.3, 0.3, 0.1, 0.1]   # 左上角小框
]
labels = [1, 2]
# 类别映射示例，可根据自己的 label_mapping 调整
label_names = {1: "class1", 2: "class2"}

# 使用之前保存的某一图像（此处示例用随机生成的灰度图）
import numpy as np
dummy_image = (np.random.rand(600, 800) * 255).astype(np.uint8)  # 注意: 此处尺寸为800宽,600高
'''

image_tensor, boxes, labels = dataset[0]

# 把 Tensor 转回 PIL，再转成 numpy array（visualize_annotations 接受 PIL 或 ndarray 都行）
to_pil = transforms.ToPILImage()
pil_img = to_pil(image_tensor)
dummy_image = np.array(pil_img)   # shape: (H, W) 或者 (H, W, C)

# 如果你的 boxes、labels 是 torch.Tensor，转成 numpy
#boxes = boxes.numpy()        # shape: (N,4)
#labels = labels.numpy()      # shape: (N,)
label_names= label_mapping
# 可视化
visualize_annotations(dummy_image, boxes, labels, label_names)
